
# Hotel Review Study — PLS-SEM Analysis Notebook

This notebook is written in a **journal-style PLS-SEM reporting structure**:

1. Data import and screening  
2. Measurement model assessment  
3. Discriminant validity  
4. Structural model assessment  
5. Bootstrapping  
6. Mediation analysis  
7. Predictive assessment  
8. IPMA-style importance-performance map  
9. Export all manuscript-ready tables and figures

> Review type is coded as: **1 = real review condition**, **0 = fake review condition**.  
> All reverse-coded items are transformed before analysis.


In [ ]:

# ============================================================
# 0. Packages and global settings
# ============================================================

import os
import re
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

RANDOM_STATE = 2026
np.random.seed(RANDOM_STATE)

# Bootstrap settings:
# Journal articles often use 5,000 bootstrap subsamples.
# If your computer is slow, temporarily change this to 500 or 1,000.
BOOTSTRAP_SAMPLES = 5000

# Screening settings
APPLY_SCREENING = True
FILTER_ADULT = True
FILTER_ONLINE_EXPERIENCE = True
FILTER_ATTENTION_CHECK = True  # will be skipped automatically if no attention-check column exists

# PLS algorithm settings
MAX_ITER = 500
TOL = 1e-7


In [ ]:

# ============================================================
# 1. File paths
# ============================================================

# Your Windows paths
USER_REAL_FILE = Path(r"D:\UserData\Desktop\学校课程\Tsui\methodology\真358295789_按序号_网络评论对消费者酒店预订决策影响问卷_34_34.xlsx")
USER_FAKE_FILE = Path(r"D:\UserData\Desktop\学校课程\Tsui\methodology\假358381191_按序号_网络评论对消费者酒店预定决策影响_38_38.xlsx")
USER_OUTPUT_DIR = Path(r"D:\UserData\Desktop\学校课程\Tsui\methodology\新版SEM结果")

# Sandbox fallback paths, useful when running inside ChatGPT/Jupyter sandbox
SANDBOX_REAL_FILE = Path(r"/mnt/data/真358295789_按序号_网络评论对消费者酒店预订决策影响问卷_34_34(1).xlsx")
SANDBOX_FAKE_FILE = Path(r"/mnt/data/假358381191_按序号_网络评论对消费者酒店预定决策影响_38_38(1).xlsx")
SANDBOX_OUTPUT_DIR = Path(r"/mnt/data/新版SEM结果")


def pick_file(user_path: Path, sandbox_path: Path) -> Path:
    """Use the Windows path on your own computer; otherwise use the sandbox file."""
    if os.name == "nt" and user_path.exists():
        return user_path
    if sandbox_path.exists():
        return sandbox_path
    if user_path.exists():
        return user_path
    raise FileNotFoundError(f"Cannot find file:\n1) {user_path}\n2) {sandbox_path}")

REAL_FILE = pick_file(USER_REAL_FILE, SANDBOX_REAL_FILE)
FAKE_FILE = pick_file(USER_FAKE_FILE, SANDBOX_FAKE_FILE)

if os.name == "nt":
    OUTPUT_DIR = USER_OUTPUT_DIR
else:
    OUTPUT_DIR = SANDBOX_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Real-review file:", REAL_FILE)
print("Fake-review file:", FAKE_FILE)
print("Output folder:", OUTPUT_DIR)



## 2. Construct design

The model uses a reflective measurement logic for the psychological constructs.

| Construct | Meaning | Items |
|---|---|---|
| ReviewType | Experimental condition | 1 = real review, 0 = fake review |
| PreBookingIntention | Booking intention before reading reviews | Q4, Q5, Q6, Q7R |
| ReviewAuthenticity | Perceived review authenticity | Q8, Q9R, Q10R, Q11 |
| Trust | Trust in the hotel after reading reviews | Q12, Q13, Q14, Q15R |
| PerceivedRisk | Perceived risk after reading reviews | Q16, Q17, Q18, Q19 |
| BookingIntention | Booking intention after reading reviews | Q20, Q21, Q22, Q23, Q24R, Q25, Q26 |

R means reverse-coded item. Since the questionnaire uses a 1–5 Likert scale, reverse coding is calculated as:

$$x_{reversed}=6-x$$


In [ ]:

# ============================================================
# 3. Data loading and preprocessing
# ============================================================

def read_condition_file(path: Path, condition_name: str, review_type: int) -> pd.DataFrame:
    df = pd.read_excel(path)
    df["Condition"] = condition_name
    df["ReviewType"] = review_type
    return df

real_raw = read_condition_file(REAL_FILE, "Real review", 1)
fake_raw = read_condition_file(FAKE_FILE, "Fake review", 0)
raw = pd.concat([real_raw, fake_raw], ignore_index=True)

print("Raw data shape:", raw.shape)
print(raw["Condition"].value_counts())


def rename_question_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename columns such as '8、我认为...' to 'Q8'. Non-question columns remain unchanged."""
    rename_dict = {}
    for col in df.columns:
        text = str(col).strip()
        m = re.match(r"^(\d+)\s*[、\.．]", text)
        if m:
            rename_dict[col] = f"Q{int(m.group(1))}"
    return df.rename(columns=rename_dict)

raw_q = rename_question_columns(raw)

# Convert question columns to numeric when possible
q_cols = [c for c in raw_q.columns if re.match(r"^Q\d+$", str(c))]
for c in q_cols:
    raw_q[c] = pd.to_numeric(raw_q[c], errors="coerce")

print("Question columns detected:", q_cols)
print("Data shape after renaming:", raw_q.shape)


In [ ]:

# ============================================================
# 4. Screening and sample description
# ============================================================

df = raw_q.copy()

screening_log = []
screening_log.append({"Step": "Initial sample", "N": len(df)})

if APPLY_SCREENING:
    if FILTER_ADULT and "Q1" in df.columns:
        df = df[df["Q1"] == 1].copy()
        screening_log.append({"Step": "After adult screening: Q1 == 1", "N": len(df)})

    if FILTER_ONLINE_EXPERIENCE and "Q2" in df.columns:
        df = df[df["Q2"] == 1].copy()
        screening_log.append({"Step": "After online shopping/booking experience screening: Q2 == 1", "N": len(df)})

    # Attention check is optional. The exported file may not include it.
    attention_candidates = [c for c in df.columns if "注意力" in str(c) or "attention" in str(c).lower()]
    if FILTER_ATTENTION_CHECK and attention_candidates:
        att_col = attention_candidates[0]
        df = df[df[att_col] == 4].copy()
        screening_log.append({"Step": f"After attention check: {att_col} == 4", "N": len(df)})

screening_table = pd.DataFrame(screening_log)
df = df.reset_index(drop=True)

print(screening_table)
print("\nFinal sample by condition:")
print(df["Condition"].value_counts())


In [ ]:

# ============================================================
# 5. Build analysis dataset and reverse-code items
# ============================================================

LIKERT_MIN = 1
LIKERT_MAX = 5
REVERSE_SUM = LIKERT_MIN + LIKERT_MAX

def reverse_likert(series: pd.Series) -> pd.Series:
    return REVERSE_SUM - series

# Item mapping from questionnaire item number to short item name.
# R items are reverse-coded before being added to the analysis dataset.
item_map = {
    # Pre-review booking intention
    "PRE1": "Q4",
    "PRE2": "Q5",
    "PRE3": "Q6",
    "PRE4": "Q7_R",

    # Perceived review authenticity
    "AUTH1": "Q8",
    "AUTH2": "Q9_R",
    "AUTH3": "Q10_R",
    "AUTH4": "Q11",

    # Trust
    "TRUST1": "Q12",
    "TRUST2": "Q13",
    "TRUST3": "Q14",
    "TRUST4": "Q15_R",

    # Perceived risk
    "RISK1": "Q16",
    "RISK2": "Q17",
    "RISK3": "Q18",
    "RISK4": "Q19",

    # Post-review booking intention
    "BI1": "Q20",
    "BI2": "Q21",
    "BI3": "Q22",
    "BI4": "Q23",
    "BI5": "Q24_R",
    "BI6": "Q25",
    "BI7": "Q26",

    # Optional decision-change indicators, not used in the core model by default
    "DC1": "Q27",
    "DC2": "Q28",
    "DC3": "Q29_R",
    "DC4": "Q30",
}

# Create reverse-coded raw columns
reverse_sources = {
    "Q7_R": "Q7",
    "Q9_R": "Q9",
    "Q10_R": "Q10",
    "Q15_R": "Q15",
    "Q24_R": "Q24",
    "Q29_R": "Q29",
}

for new_col, old_col in reverse_sources.items():
    if old_col in df.columns:
        df[new_col] = reverse_likert(df[old_col])

analysis = pd.DataFrame({
    "Condition": df["Condition"],
    "ReviewType": df["ReviewType"],
})

missing_items = []
for short_name, source_col in item_map.items():
    if source_col in df.columns:
        analysis[short_name] = pd.to_numeric(df[source_col], errors="coerce")
    else:
        missing_items.append((short_name, source_col))

if missing_items:
    print("Warning: missing items:", missing_items)

# Optional control variables / descriptive variables
control_map = {
    "ReviewReadingFrequency": "Q3",
    "BehaviorChoice": "Q31",
    "WillingnessToPay": "Q32",
    "DecisionTime": "Q33",
    "SwitchReason": "Q34",
    "ReviewReliance": "Q35",
    "RatingImportance": "Q36",
    "FakeReviewExperience": "Q37",
    "FakeReviewAbility": "Q38",
    "ReviewSkepticism": "Q39",
    "RecentOnlineSatisfaction": "Q40",
    "ReviewPostingHabit": "Q41",
    "Gender": "Q42",
    "Age": "Q43",
    "Education": "Q44",
    "MonthlyOnlineFrequency": "Q45",
    "DisposableIncome": "Q46",
}

for new_col, old_col in control_map.items():
    if old_col in df.columns:
        analysis[new_col] = pd.to_numeric(df[old_col], errors="coerce")

print("Analysis data shape:", analysis.shape)
display(analysis.head())


In [ ]:

# ============================================================
# 6. Measurement blocks and structural model
# ============================================================

# Core PLS-SEM measurement blocks
blocks = {
    "ReviewType": ["ReviewType"],
    "PreBookingIntention": ["PRE1", "PRE2", "PRE3", "PRE4"],
    "ReviewAuthenticity": ["AUTH1", "AUTH2", "AUTH3", "AUTH4"],
    "Trust": ["TRUST1", "TRUST2", "TRUST3", "TRUST4"],
    "PerceivedRisk": ["RISK1", "RISK2", "RISK3", "RISK4"],
    "BookingIntention": ["BI1", "BI2", "BI3", "BI4", "BI5", "BI6", "BI7"],
}

# Structural paths: target construct -> predictor constructs
path_model = {
    "ReviewAuthenticity": ["ReviewType"],
    "Trust": ["ReviewType", "ReviewAuthenticity"],
    "PerceivedRisk": ["ReviewType", "ReviewAuthenticity"],
    "BookingIntention": ["ReviewType", "PreBookingIntention", "ReviewAuthenticity", "Trust", "PerceivedRisk"],
}

construct_order = list(blocks.keys())

# Check that all indicators exist
all_indicators = [item for items in blocks.values() for item in items]
missing = [c for c in all_indicators if c not in analysis.columns]
if missing:
    raise ValueError(f"Missing indicators in analysis data: {missing}")

pls_data = analysis[all_indicators].copy()
pls_data = pls_data.apply(pd.to_numeric, errors="coerce")

# Mean imputation for small accidental missingness. This is common in exploratory small-sample analysis.
pls_data = pls_data.fillna(pls_data.mean(numeric_only=True))

print("PLS data shape:", pls_data.shape)
print("Constructs:", construct_order)


In [ ]:

# ============================================================
# 7. Helper functions for PLS-SEM
# ============================================================

def zscore_df(df: pd.DataFrame) -> pd.DataFrame:
    return (df - df.mean()) / df.std(ddof=0)


def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def cronbach_alpha(X: pd.DataFrame) -> float:
    X = X.dropna()
    k = X.shape[1]
    if k < 2:
        return np.nan
    variances = X.var(axis=0, ddof=1)
    total_var = X.sum(axis=1).var(ddof=1)
    if total_var <= 0:
        return np.nan
    return float(k / (k - 1) * (1 - variances.sum() / total_var))


def composite_reliability(loadings: np.ndarray) -> float:
    loadings = np.asarray(loadings, dtype=float)
    if len(loadings) == 1:
        return 1.0
    numerator = np.sum(loadings) ** 2
    denominator = numerator + np.sum(1 - loadings ** 2)
    return float(numerator / denominator) if denominator != 0 else np.nan


def ave(loadings: np.ndarray) -> float:
    loadings = np.asarray(loadings, dtype=float)
    if len(loadings) == 1:
        return 1.0
    return float(np.mean(loadings ** 2))


def rho_a_approx(X: pd.DataFrame) -> float:
    """Approximate rho_A using a one-factor PCA reliability approximation.
    This is not a replacement for SmartPLS rho_A, but gives a manuscript-style reliability column.
    """
    X = zscore_df(X).dropna()
    k = X.shape[1]
    if k < 2:
        return np.nan
    R = np.corrcoef(X.values, rowvar=False)
    vals, vecs = np.linalg.eigh(R)
    loading = vecs[:, -1] * np.sqrt(max(vals[-1], 0))
    if np.sum(loading) < 0:
        loading = -loading
    numerator = np.sum(loading) ** 2
    denominator = numerator + np.sum(np.maximum(0, 1 - loading ** 2))
    return float(numerator / denominator) if denominator != 0 else np.nan


def ols_standardized(y: np.ndarray, X: np.ndarray):
    """OLS without intercept because PLS latent scores are standardized."""
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_hat = X @ beta
    sse = np.sum((y - y_hat) ** 2)
    sst = np.sum((y - y.mean()) ** 2)
    r2 = 1 - sse / sst if sst > 0 else np.nan
    return beta, y_hat, float(r2)


def inner_connections(blocks, path_model):
    con = {c: set() for c in blocks}
    for target, preds in path_model.items():
        for pred in preds:
            con[target].add(pred)
            con[pred].add(target)
    return con


def run_pls_pm(data: pd.DataFrame, blocks: dict, path_model: dict,
               max_iter: int = MAX_ITER, tol: float = TOL):
    """A lightweight reflective Mode-A PLS-PM implementation.

    It estimates latent variable scores through an iterative outer-weight algorithm,
    then estimates structural paths using standardized OLS on latent scores.
    """
    # Standardized indicators
    X_all = zscore_df(data.copy()).replace([np.inf, -np.inf], np.nan)
    X_all = X_all.fillna(0.0)

    # Initial weights
    weights = {}
    for construct, items in blocks.items():
        k = len(items)
        if k == 1:
            weights[construct] = np.array([1.0])
        else:
            weights[construct] = np.ones(k) / np.sqrt(k)

    connections = inner_connections(blocks, path_model)
    last_weights = {k: v.copy() for k, v in weights.items()}

    for iteration in range(max_iter):
        # Outer approximation: latent variable scores
        scores = {}
        for construct, items in blocks.items():
            X = X_all[items].values
            y = X @ weights[construct]
            std = np.std(y)
            if std > 1e-12:
                y = (y - np.mean(y)) / std
            else:
                y = y - np.mean(y)
            scores[construct] = y

        # Inner approximation using centroid scheme
        inner_estimates = {}
        for construct in blocks:
            connected = list(connections[construct])
            if not connected:
                inner_estimates[construct] = scores[construct]
                continue
            z = np.zeros(len(data))
            for other in connected:
                sign = np.sign(safe_corr(scores[construct], scores[other]))
                if sign == 0:
                    sign = 1
                z += sign * scores[other]
            if np.std(z) > 1e-12:
                z = (z - np.mean(z)) / np.std(z)
            inner_estimates[construct] = z

        # Update outer weights, reflective Mode A
        for construct, items in blocks.items():
            if len(items) == 1:
                weights[construct] = np.array([1.0])
                continue
            X = X_all[items].values
            z = inner_estimates[construct]
            new_w = np.array([safe_corr(X[:, j], z) for j in range(X.shape[1])])
            norm = np.linalg.norm(new_w)
            if norm < 1e-12:
                new_w = last_weights[construct]
            else:
                new_w = new_w / norm
            weights[construct] = new_w

        # Check convergence
        max_change = max(np.max(np.abs(weights[c] - last_weights[c])) for c in blocks)
        if max_change < tol:
            break
        last_weights = {k: v.copy() for k, v in weights.items()}

    # Final scores
    scores = {}
    loadings = {}
    for construct, items in blocks.items():
        X = X_all[items].values
        y = X @ weights[construct]
        if np.std(y) > 1e-12:
            y = (y - np.mean(y)) / np.std(y)
        scores[construct] = y

        # Outer loadings
        lds = np.array([safe_corr(X[:, j], y) for j in range(X.shape[1])])
        # Align sign so that most loadings are positive
        if np.nanmean(lds) < 0:
            y = -y
            scores[construct] = y
            weights[construct] = -weights[construct]
            lds = -lds
        loadings[construct] = lds

    score_df = pd.DataFrame(scores)

    # Structural model
    paths = []
    r2 = {}
    predictions = {}
    for target, preds in path_model.items():
        y = score_df[target].values
        X = score_df[preds].values
        beta, y_hat, target_r2 = ols_standardized(y, X)
        r2[target] = target_r2
        predictions[target] = y_hat
        for pred, b in zip(preds, beta):
            paths.append({"Path": f"{pred} -> {target}", "Predictor": pred, "Target": target, "Beta": float(b)})

    paths_df = pd.DataFrame(paths)
    r2_df = pd.DataFrame([{"Construct": k, "R2": v, "Adjusted_R2": 1 - (1 - v) * (len(score_df) - 1) / (len(score_df) - len(path_model[k]) - 1)}
                          for k, v in r2.items()])

    return {
        "scores": score_df,
        "weights": weights,
        "loadings": loadings,
        "paths": paths_df,
        "r2": r2_df,
        "iterations": iteration + 1,
        "predictions": predictions,
    }


In [ ]:

# ============================================================
# 8. Run PLS-SEM model
# ============================================================

pls_result = run_pls_pm(pls_data, blocks, path_model)
scores = pls_result["scores"]
paths_original = pls_result["paths"].copy()
r2_original = pls_result["r2"].copy()

print("PLS algorithm iterations:", pls_result["iterations"])
print("\nLatent variable score descriptives:")
display(scores.describe().T)
print("\nOriginal path coefficients:")
display(paths_original)
print("\nR-squared:")
display(r2_original)


In [ ]:

# ============================================================
# 9. Measurement model assessment
# ============================================================

# Outer loadings table
loading_rows = []
for construct, items in blocks.items():
    for item, loading in zip(items, pls_result["loadings"][construct]):
        loading_rows.append({
            "Construct": construct,
            "Indicator": item,
            "Outer_Loading": loading,
            "Loading_Squared": loading ** 2,
            "Recommended_Loading_0.708": "Pass" if abs(loading) >= 0.708 else "Check"
        })
outer_loadings = pd.DataFrame(loading_rows)

# Reliability and convergent validity
reliability_rows = []
for construct, items in blocks.items():
    X = pls_data[items]
    lds = pls_result["loadings"][construct]
    if len(items) == 1:
        alpha = np.nan
        rho_a = np.nan
        cr = 1.0
        ave_value = 1.0
    else:
        alpha = cronbach_alpha(X)
        rho_a = rho_a_approx(X)
        cr = composite_reliability(lds)
        ave_value = ave(lds)
    reliability_rows.append({
        "Construct": construct,
        "Items": len(items),
        "Cronbach_Alpha": alpha,
        "rho_A_Approx": rho_a,
        "Composite_Reliability": cr,
        "AVE": ave_value,
        "Alpha_0.70": "Pass" if (np.isnan(alpha) or alpha >= 0.70) else "Check",
        "CR_0.70": "Pass" if cr >= 0.70 else "Check",
        "AVE_0.50": "Pass" if ave_value >= 0.50 else "Check",
    })
measurement_model = pd.DataFrame(reliability_rows)

print("Outer loadings:")
display(outer_loadings)
print("\nReliability and convergent validity:")
display(measurement_model)


In [ ]:

# ============================================================
# 10. Discriminant validity: Fornell-Larcker, cross-loadings, HTMT
# ============================================================

# Fornell-Larcker criterion
lv_corr = scores.corr()
ave_lookup = measurement_model.set_index("Construct")["AVE"].to_dict()
fornell = lv_corr.copy()
for c in fornell.index:
    if c in ave_lookup:
        fornell.loc[c, c] = np.sqrt(ave_lookup[c])

# Cross-loadings: indicator correlations with LV scores
cross_loading_rows = []
for item in all_indicators:
    for construct in construct_order:
        cross_loading_rows.append({
            "Indicator": item,
            "Construct": construct,
            "Cross_Loading": safe_corr(pls_data[item], scores[construct]),
            "Primary_Construct": next((k for k, v in blocks.items() if item in v), None)
        })
cross_loadings = pd.DataFrame(cross_loading_rows)

# HTMT ratio
indicator_corr = pls_data.corr().abs()
htmt = pd.DataFrame(index=construct_order, columns=construct_order, dtype=float)
for a in construct_order:
    for b in construct_order:
        if a == b:
            htmt.loc[a, b] = 1.0
        else:
            items_a = blocks[a]
            items_b = blocks[b]
            if len(items_a) < 2 or len(items_b) < 2:
                htmt.loc[a, b] = np.nan
                continue
            heterotrait = indicator_corr.loc[items_a, items_b].values
            # monotrait correlations excluding diagonal
            mono_a = indicator_corr.loc[items_a, items_a].values
            mono_b = indicator_corr.loc[items_b, items_b].values
            mono_a_vals = mono_a[np.triu_indices_from(mono_a, k=1)]
            mono_b_vals = mono_b[np.triu_indices_from(mono_b, k=1)]
            denom = np.sqrt(np.mean(mono_a_vals) * np.mean(mono_b_vals))
            htmt.loc[a, b] = np.mean(heterotrait) / denom if denom > 0 else np.nan

print("Fornell-Larcker criterion: diagonal = sqrt(AVE), off-diagonal = construct correlations")
display(fornell)
print("\nHTMT ratio:")
display(htmt)
print("\nCross-loadings:")
display(cross_loadings.pivot(index="Indicator", columns="Construct", values="Cross_Loading"))


In [ ]:

# ============================================================
# 11. Structural model: VIF, f2, Q2_predict
# ============================================================

def predictor_vif(score_df: pd.DataFrame, preds: list) -> pd.DataFrame:
    rows = []
    if len(preds) < 2:
        return pd.DataFrame([{"Predictor": preds[0], "VIF": 1.0}]) if preds else pd.DataFrame()
    for pred in preds:
        others = [p for p in preds if p != pred]
        y = score_df[pred].values
        X = score_df[others].values
        _, _, r2 = ols_standardized(y, X)
        vif = 1 / (1 - r2) if r2 < 1 else np.inf
        rows.append({"Predictor": pred, "VIF": vif})
    return pd.DataFrame(rows)

vif_rows = []
for target, preds in path_model.items():
    tmp = predictor_vif(scores, preds)
    tmp["Target"] = target
    vif_rows.append(tmp)
inner_vif = pd.concat(vif_rows, ignore_index=True)
inner_vif = inner_vif[["Target", "Predictor", "VIF"]]
inner_vif["VIF_3.3"] = np.where(inner_vif["VIF"] <= 3.3, "Pass", "Check")
inner_vif["VIF_5.0"] = np.where(inner_vif["VIF"] <= 5.0, "Pass", "Check")

# f-square effect size
f2_rows = []
for target, preds in path_model.items():
    y = scores[target].values
    X_full = scores[preds].values
    _, _, r2_full = ols_standardized(y, X_full)
    for pred in preds:
        reduced_preds = [p for p in preds if p != pred]
        if reduced_preds:
            _, _, r2_reduced = ols_standardized(y, scores[reduced_preds].values)
        else:
            r2_reduced = 0.0
        f2 = (r2_full - r2_reduced) / (1 - r2_full) if r2_full < 1 else np.nan
        if pd.isna(f2):
            size = "NA"
        elif f2 >= 0.35:
            size = "Large"
        elif f2 >= 0.15:
            size = "Medium"
        elif f2 >= 0.02:
            size = "Small"
        else:
            size = "Very small"
        f2_rows.append({"Target": target, "Predictor": pred, "R2_Full": r2_full, "R2_Reduced": r2_reduced, "f2": f2, "Effect_Size": size})
f2_table = pd.DataFrame(f2_rows)

# Cross-validated predictive relevance using K-fold prediction on LV scores
q2_rows = []
K = min(5, len(scores))
kf = KFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)
for target, preds in path_model.items():
    y = scores[target].values
    y_pred = np.full(len(y), np.nan)
    for train_idx, test_idx in kf.split(scores):
        X_train = scores.iloc[train_idx][preds].values
        y_train = y[train_idx]
        X_test = scores.iloc[test_idx][preds].values
        beta, _, _ = ols_standardized(y_train, X_train)
        y_pred[test_idx] = X_test @ beta
    press = np.sum((y - y_pred) ** 2)
    sse_mean = np.sum((y - y.mean()) ** 2)
    q2_predict = 1 - press / sse_mean if sse_mean > 0 else np.nan
    rmse = np.sqrt(np.mean((y - y_pred) ** 2))
    mae = np.mean(np.abs(y - y_pred))
    q2_rows.append({"Construct": target, "Q2_predict": q2_predict, "RMSE": rmse, "MAE": mae})
q2_predict = pd.DataFrame(q2_rows)

print("Inner VIF:")
display(inner_vif)
print("\nf-square effect sizes:")
display(f2_table)
print("\nPredictive assessment:")
display(q2_predict)


In [ ]:

# ============================================================
# 12. Bootstrapping for path coefficients and indirect effects
# ============================================================

def path_series_from_result(result):
    s = result["paths"].set_index("Path")["Beta"]
    return s

original_path_series = path_series_from_result(pls_result)
path_names = list(original_path_series.index)

# Indirect-effect specifications
# Each indirect effect is a product of path coefficients.
indirect_specs = {
    "ReviewType -> ReviewAuthenticity -> BookingIntention": [
        "ReviewType -> ReviewAuthenticity",
        "ReviewAuthenticity -> BookingIntention",
    ],
    "ReviewType -> ReviewAuthenticity -> Trust -> BookingIntention": [
        "ReviewType -> ReviewAuthenticity",
        "ReviewAuthenticity -> Trust",
        "Trust -> BookingIntention",
    ],
    "ReviewType -> ReviewAuthenticity -> PerceivedRisk -> BookingIntention": [
        "ReviewType -> ReviewAuthenticity",
        "ReviewAuthenticity -> PerceivedRisk",
        "PerceivedRisk -> BookingIntention",
    ],
    "ReviewType -> Trust -> BookingIntention": [
        "ReviewType -> Trust",
        "Trust -> BookingIntention",
    ],
    "ReviewType -> PerceivedRisk -> BookingIntention": [
        "ReviewType -> PerceivedRisk",
        "PerceivedRisk -> BookingIntention",
    ],
    "ReviewAuthenticity -> Trust -> BookingIntention": [
        "ReviewAuthenticity -> Trust",
        "Trust -> BookingIntention",
    ],
    "ReviewAuthenticity -> PerceivedRisk -> BookingIntention": [
        "ReviewAuthenticity -> PerceivedRisk",
        "PerceivedRisk -> BookingIntention",
    ],
}


def compute_indirect_effects(path_series: pd.Series) -> pd.Series:
    values = {}
    for name, paths in indirect_specs.items():
        product = 1.0
        ok = True
        for p in paths:
            if p not in path_series.index:
                ok = False
                break
            product *= path_series[p]
        values[name] = product if ok else np.nan
    return pd.Series(values)

original_indirect_series = compute_indirect_effects(original_path_series)

rng = np.random.default_rng(RANDOM_STATE)
boot_paths = []
boot_indirect = []
failed = 0
n = len(pls_data)

print(f"Starting bootstrapping: B = {BOOTSTRAP_SAMPLES}, N = {n}")
for b in range(BOOTSTRAP_SAMPLES):
    idx = rng.integers(0, n, size=n)
    sample = pls_data.iloc[idx].reset_index(drop=True)
    try:
        res_b = run_pls_pm(sample, blocks, path_model, max_iter=MAX_ITER, tol=TOL)
        ps = path_series_from_result(res_b)
        boot_paths.append(ps.reindex(path_names))
        boot_indirect.append(compute_indirect_effects(ps).reindex(original_indirect_series.index))
    except Exception as e:
        failed += 1
    if (b + 1) % max(1, BOOTSTRAP_SAMPLES // 10) == 0:
        print(f"  Bootstrap progress: {b + 1}/{BOOTSTRAP_SAMPLES}")

boot_paths_df = pd.DataFrame(boot_paths)
boot_indirect_df = pd.DataFrame(boot_indirect)
print("Bootstrap completed. Failed samples:", failed)


def bootstrap_summary(original: pd.Series, boot_df: pd.DataFrame, name_col: str) -> pd.DataFrame:
    rows = []
    for name in original.index:
        boot_vals = pd.to_numeric(boot_df[name], errors="coerce").dropna().values
        orig = original[name]
        if len(boot_vals) < 5:
            se = t_value = p_value = ci_low = ci_high = np.nan
        else:
            se = np.std(boot_vals, ddof=1)
            t_value = orig / se if se > 0 else np.nan
            p_value = 2 * (1 - stats.t.cdf(abs(t_value), df=len(boot_vals)-1)) if not pd.isna(t_value) else np.nan
            ci_low, ci_high = np.percentile(boot_vals, [2.5, 97.5])
        rows.append({
            name_col: name,
            "Original_Effect": orig,
            "Bootstrap_Mean": np.mean(boot_vals) if len(boot_vals) else np.nan,
            "Bootstrap_SD": se,
            "t_value": t_value,
            "p_value": p_value,
            "CI_2.5%": ci_low,
            "CI_97.5%": ci_high,
            "Significant_0.05": "Yes" if (not pd.isna(p_value) and p_value < 0.05) else "No"
        })
    return pd.DataFrame(rows)

structural_paths = bootstrap_summary(original_path_series, boot_paths_df, "Path")
mediation_effects = bootstrap_summary(original_indirect_series, boot_indirect_df, "Indirect_Effect")

print("Structural path results:")
display(structural_paths)
print("\nMediation results:")
display(mediation_effects)


In [ ]:

# ============================================================
# 13. Group descriptives and condition comparison
# ============================================================

# Construct mean scores based on raw items, before z-scoring, for descriptive reporting
raw_construct_scores = pd.DataFrame({
    "Condition": analysis["Condition"],
    "ReviewType": analysis["ReviewType"],
})
for construct, items in blocks.items():
    if construct == "ReviewType":
        raw_construct_scores[construct] = analysis["ReviewType"]
    else:
        raw_construct_scores[construct] = analysis[items].mean(axis=1)

# Keep only screened rows by aligning index; analysis was built from screened df, so this is okay.
group_desc = raw_construct_scores.groupby("Condition")[construct_order].agg(["count", "mean", "std"]).round(4)

# Welch t-tests between real and fake conditions for construct scores
condition_tests = []
for construct in construct_order:
    if construct == "ReviewType":
        continue
    real_vals = raw_construct_scores.loc[raw_construct_scores["Condition"] == "Real review", construct].dropna()
    fake_vals = raw_construct_scores.loc[raw_construct_scores["Condition"] == "Fake review", construct].dropna()
    t_stat, p_val = stats.ttest_ind(real_vals, fake_vals, equal_var=False, nan_policy="omit")
    mean_diff = real_vals.mean() - fake_vals.mean()
    # Cohen's d using pooled sd
    pooled_sd = np.sqrt(((len(real_vals)-1)*real_vals.var(ddof=1) + (len(fake_vals)-1)*fake_vals.var(ddof=1)) / (len(real_vals)+len(fake_vals)-2))
    d = mean_diff / pooled_sd if pooled_sd > 0 else np.nan
    condition_tests.append({
        "Construct": construct,
        "Real_Mean": real_vals.mean(),
        "Fake_Mean": fake_vals.mean(),
        "Mean_Difference_Real_minus_Fake": mean_diff,
        "t_value": t_stat,
        "p_value": p_val,
        "Cohens_d": d,
    })
condition_comparison = pd.DataFrame(condition_tests)

print("Group descriptives:")
display(group_desc)
print("\nCondition comparison:")
display(condition_comparison)


In [ ]:

# ============================================================
# 14. Importance-performance map analysis, IPMA-style
# ============================================================

def total_effects_matrix(paths_df: pd.DataFrame, constructs: list) -> pd.DataFrame:
    # B[target, predictor] = beta
    B = pd.DataFrame(0.0, index=constructs, columns=constructs)
    for _, row in paths_df.iterrows():
        B.loc[row["Target"], row["Predictor"]] = row["Beta"]
    I = np.eye(len(constructs))
    try:
        total = np.linalg.inv(I - B.values) - I
    except np.linalg.LinAlgError:
        total = np.full_like(B.values, np.nan, dtype=float)
    return pd.DataFrame(total, index=constructs, columns=constructs)

total_effects = total_effects_matrix(paths_original, construct_order)

target_construct = "BookingIntention"
ipma_rows = []
for construct in construct_order:
    if construct == target_construct:
        continue
    importance = total_effects.loc[target_construct, construct]
    if construct == "ReviewType":
        performance = raw_construct_scores[construct].mean() * 100
    else:
        performance = (raw_construct_scores[construct].mean() - LIKERT_MIN) / (LIKERT_MAX - LIKERT_MIN) * 100
    ipma_rows.append({
        "Target": target_construct,
        "Construct": construct,
        "Importance_Total_Effect": importance,
        "Performance_0_100": performance,
    })
ipma_table = pd.DataFrame(ipma_rows).sort_values("Importance_Total_Effect", ascending=False)

print("Total effects matrix:")
display(total_effects)
print("\nIPMA-style table:")
display(ipma_table)

# IPMA plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ipma_table["Performance_0_100"], ipma_table["Importance_Total_Effect"], s=90)
for _, row in ipma_table.iterrows():
    ax.annotate(row["Construct"], (row["Performance_0_100"], row["Importance_Total_Effect"]),
                textcoords="offset points", xytext=(6, 6), fontsize=9)
ax.axvline(ipma_table["Performance_0_100"].mean(), linestyle="--", linewidth=1)
ax.axhline(ipma_table["Importance_Total_Effect"].mean(), linestyle="--", linewidth=1)
ax.set_xlabel("Performance, 0–100")
ax.set_ylabel("Importance, total effect on BookingIntention")
ax.set_title("Importance-Performance Map Analysis, IPMA-style")
ax.grid(True, alpha=0.3)
plt.tight_layout()
ipma_fig_path = OUTPUT_DIR / "IPMA_BookingIntention.png"
plt.savefig(ipma_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", ipma_fig_path)


In [ ]:

# ============================================================
# 15. Path diagram
# ============================================================

# Add significance stars to structural paths
path_sig = structural_paths.set_index("Path")

def star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return "†"
    return ""

coords = {
    "ReviewType": (0.05, 0.65),
    "PreBookingIntention": (0.05, 0.25),
    "ReviewAuthenticity": (0.36, 0.65),
    "Trust": (0.62, 0.82),
    "PerceivedRisk": (0.62, 0.48),
    "BookingIntention": (0.90, 0.65),
}

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# Draw nodes
for construct, (x, y) in coords.items():
    ax.text(x, y, construct, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.2), fontsize=10)

# Draw arrows
for _, row in paths_original.iterrows():
    pred, target, beta = row["Predictor"], row["Target"], row["Beta"]
    if pred not in coords or target not in coords:
        continue
    x1, y1 = coords[pred]
    x2, y2 = coords[target]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=1.4, shrinkA=22, shrinkB=22))
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    pval = path_sig.loc[row["Path"], "p_value"] if row["Path"] in path_sig.index else np.nan
    ax.text(mx, my + 0.025, f"β={beta:.3f}{star(pval)}", ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))

# R2 labels
for _, row in r2_original.iterrows():
    construct = row["Construct"]
    if construct in coords:
        x, y = coords[construct]
        ax.text(x, y - 0.085, f"R²={row['R2']:.3f}", ha="center", va="center", fontsize=9)

ax.set_title("PLS-SEM Structural Model", fontsize=14, pad=18)
plt.tight_layout()
path_fig_path = OUTPUT_DIR / "PLS_SEM_Path_Diagram.png"
plt.savefig(path_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", path_fig_path)


In [ ]:

# ============================================================
# 16. Export all tables and figures
# ============================================================

# Combine R2, f2, and Q2 into manuscript tables
r2_q2 = r2_original.merge(q2_predict, left_on="Construct", right_on="Construct", how="left")

# Correlations and descriptives
construct_correlations = scores.corr()
construct_descriptives = raw_construct_scores[construct_order].describe().T

# Save cleaned datasets
analysis.to_csv(OUTPUT_DIR / "cleaned_analysis_dataset.csv", index=False, encoding="utf-8-sig")
scores.to_csv(OUTPUT_DIR / "latent_variable_scores.csv", index=False, encoding="utf-8-sig")

# Main Excel workbook
excel_path = OUTPUT_DIR / "PLS_SEM_Manuscript_Tables.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    screening_table.to_excel(writer, sheet_name="Screening", index=False)
    measurement_model.to_excel(writer, sheet_name="Reliability_AVE", index=False)
    outer_loadings.to_excel(writer, sheet_name="Outer_Loadings", index=False)
    fornell.to_excel(writer, sheet_name="Fornell_Larcker")
    htmt.to_excel(writer, sheet_name="HTMT")
    cross_loadings.to_excel(writer, sheet_name="Cross_Loadings", index=False)
    inner_vif.to_excel(writer, sheet_name="Inner_VIF", index=False)
    structural_paths.to_excel(writer, sheet_name="Structural_Paths", index=False)
    r2_q2.to_excel(writer, sheet_name="R2_Q2", index=False)
    f2_table.to_excel(writer, sheet_name="f2_Effect_Size", index=False)
    mediation_effects.to_excel(writer, sheet_name="Mediation", index=False)
    condition_comparison.to_excel(writer, sheet_name="Condition_Comparison", index=False)
    construct_descriptives.to_excel(writer, sheet_name="Construct_Descriptives")
    construct_correlations.to_excel(writer, sheet_name="Construct_Correlations")
    total_effects.to_excel(writer, sheet_name="Total_Effects")
    ipma_table.to_excel(writer, sheet_name="IPMA", index=False)
    raw_construct_scores.to_excel(writer, sheet_name="Construct_Scores_Raw", index=False)

# Also save key tables as individual CSV files
for name, table in {
    "measurement_model": measurement_model,
    "outer_loadings": outer_loadings,
    "fornell_larcker": fornell,
    "htmt": htmt,
    "inner_vif": inner_vif,
    "structural_paths_bootstrap": structural_paths,
    "r2_q2": r2_q2,
    "f2_effect_size": f2_table,
    "mediation_effects": mediation_effects,
    "condition_comparison": condition_comparison,
    "ipma_table": ipma_table,
}.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=True, encoding="utf-8-sig")

# Save a compact JSON summary for easy checking
summary = {
    "N_final": int(len(pls_data)),
    "N_by_condition": raw_construct_scores["Condition"].value_counts().to_dict(),
    "Bootstrap_samples": int(BOOTSTRAP_SAMPLES),
    "PLS_iterations": int(pls_result["iterations"]),
    "Output_folder": str(OUTPUT_DIR),
}
with open(OUTPUT_DIR / "analysis_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("All results exported successfully.")
print("Excel workbook:", excel_path)
print("Cleaned data:", OUTPUT_DIR / "cleaned_analysis_dataset.csv")
print("Figures:")
print(" -", ipma_fig_path)
print(" -", path_fig_path)
print("Summary:", summary)



## 17. Suggested manuscript-style reporting logic

When writing the methodology/results section, the output tables can be reported in this order:

1. **Sample and data screening**: use the `Screening` sheet.  
2. **Measurement model**: report `Outer_Loadings`, `Reliability_AVE`, `Fornell_Larcker`, and `HTMT`.  
3. **Structural model**: report `Inner_VIF`, `Structural_Paths`, `R2_Q2`, and `f2_Effect_Size`.  
4. **Mediation**: report `Mediation`.  
5. **Condition comparison**: report `Condition_Comparison` only as supplementary evidence.  
6. **Managerial/practical interpretation**: use `IPMA` and `IPMA_BookingIntention.png`.

Typical threshold language:

- Outer loading: preferably above 0.708.  
- Composite reliability: above 0.70.  
- AVE: above 0.50.  
- HTMT: below 0.85 or 0.90.  
- Inner VIF: below 3.3 or 5.0.  
- Bootstrap path: significant if p < 0.05 and confidence interval does not include zero.

Because this study has a small sample, use careful language such as:

> The results provide preliminary evidence rather than definitive population-level confirmation.
